# Training a phase reconstructor for OZIRIIS

This notebook trains a convolutional neural network to reconstruct wavefronts from OZIRIIS's (vector-Zernike WFS) detector frames, using the fully differentiable end-to-end simulation.

Unlike the calibration notebook — where we fit the WFS and DM to match real bench measurements — here the WFS and DM are loaded from a previous calibration and then frozen (`requires_grad_(False)`); only the reconstructor network's weights are updated.

Training simulates an AO closed loop: at each step we draw a random turbulent phase screen, propagate the *residual* phase (after applying the previous correction) through the WFS, and backpropagate the reconstruction error through the optical model and DM straight into the network. This is implemented by `AI4AO.Trainer.Trainer` (see `AI4AO/Trainer.py`), which this notebook uses for training, evaluation, loss plotting, and checkpointing.

In [ ]:
from mmengine import Config
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import time

from AI4AO import ZernikeWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss

## Reconstructor architecture

`Papyrus2ndStage` is the CNN that maps the preprocessed pupil images (one per arm of the vector-Zernike sensor) to a vector of `Nmodes` KL-mode coefficients: a convolutional encoder that downsamples with `MaxPool2d` down to a single spatial position, followed by a linear head. Reconstructor architectures like this one are defined directly in the tutorial notebooks rather than in `AI4AO/`, so it's easy to try different networks per instrument.

In [ ]:
class Papyrus2ndStage(nn.Module):
    def __init__(self, dmParams):
        super().__init__()

        Nmodes = dmParams["Nmodes"]

        self.encoder = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=11, padding=7),
            nn.LeakyReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=7, padding=5),
            nn.LeakyReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=5, padding=3),
            nn.LeakyReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=2),
            nn.LeakyReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=2),
            nn.LeakyReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(256, 512, kernel_size=2, padding=2),
            nn.LeakyReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.1),
        )

        self.outputlayer = nn.Sequential(
            nn.Linear(512, Nmodes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.encoder(x)
        x = self.outputlayer(x)
        return x

## Loading the instrument configuration

Each instrument has a dedicated params file (here `Oziriis_params.py`) holding the plain-dict configs consumed positionally by the pipeline constructors — `WFSParams`, `AtmosParams`, `LoopParams`, `TrainParams`, `DMParams`. There is no YAML/JSON config layer in this codebase; these dicts *are* the configuration mechanism.

In [ ]:
device = 'cuda' # set to "cpu" if Cuda is not available
    
paramfile = 'Oziriis_params.py'  # file of experimental parameters


# Config extraction
AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']
DMParams = Config.fromfile(paramfile)['DMParams']

## Dataset

`PhaseDataset` generates atmospheric phase screens on the fly from a von Kármán PSD. Setting `dataset.generateClosedLoop = True` makes the sample include partially corrected wavefronts to be generated. Consecutive samples (`dataset[0]`, `dataset[1]`, ...) represent consecutive AO-loop time steps — with wind translation applied between them — instead of independent draws, which is what the closed-loop training and evaluation loops below assume. Repeated calls to dataset[0] will generate new wavefronts.

```python
batch = dataset[0] # new set of wavefronts
batch = dataset[0] # new set of wavefronts
batch = dataset[0] # new set of wavefronts

batch = dataset[1] # same wavefront as the last one, advanced one iteration in time
batch = dataset[1] # same wavefront as the last one
batch = dataset[2] # same wavefront as the last one, advanced one iteration in time
batch = dataset[12] # same wavefront as the last one, advanced ten iterations in time
```

In [ ]:
# Dataset creation
dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

## Loading the calibrated WFS and DM

We load the WFS mask geometry and DM misregistration/influence functions fit in `Oziiris_twin_calibration.ipynb`, then freeze them (`.eval()` + `requires_grad_(False)`) — in this notebook only the reconstructor network is trained, so gradients should not flow into the optical model or DM geometry. Re-centering the influence functions (masking by the pupil, then subtracting each actuator's mean over the pupil) removes DC offsets outside the aperture so DM commands don't inject unphysical piston into the reconstructed phase.

In [ ]:
PATH = "../Data/Oziriis/"
# WFS creation
wfs = ZernikeWFS(WFSParams, device)
PATH_WFS = PATH + "OziriisWFS.pth"
wfs.LoadCalibration(PATH_WFS)
wfs.eval()
wfs.requires_grad_(False)

dm = DeformableMirror(WFSParams, DMParams, device)
PATH_DM = PATH + "OziriisDM.pth"
dm.LoadCalibration(PATH_DM)
dm.eval()
dm.requires_grad_(False)
dm.IF *= dataset.pupil
dm.IF[:, dataset.pupil] -= dm.IF[:, dataset.pupil].mean(dim = (-1), keepdim = True)


## Modes-to-commands matrix

`M2C` converts a vector of modal coefficients into DM actuator commands, so `dm(M2C.T)` gives the full-resolution phase produced by each individual mode on its own. `z_inv`, its pseudo-inverse, does the reverse: projecting a full-resolution phase screen onto modal coefficients. This is how the training loop obtains the "ground truth" modal coefficients for a given turbulence phase screen, to compare against the reconstructor's prediction.

In [ ]:
M2C = np.load(PATH + "M2C_KL.npy")
M2C = torch.from_numpy(M2C).to(device = device, dtype = torch.float32)
M2C = M2C[:, :DMParams["Nmodes"]]  # keep only the modes this reconstructor is trained to output

z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim = -2))  # full-resolution phase -> modal coefficients

## WFS depth override

The vector-Zernike WFS is parameterized by the phase-shift depth of each of its two Zernike phase dots (`wfs.depths`). Here we override the depths loaded from calibration and rebuild the mask and reference intensity — useful for testing a specific configuration rather than reusing whatever the calibration procedure converged to.

In [ ]:
with torch.no_grad():
    wfs.depths[:] = torch.tensor(
        [torch.pi * -0.33, torch.pi * 0.76],
        device=wfs.depths.device
    )
    
wfs.BuildMask()
wfs.BuildReferenceIntensity()

## Frame preprocessing

`FramePreprocess` crops the individual pupil images out of the raw WFS detector frame and reference-subtracts/normalizes them before they reach the reconstructor. `ProcessReference` records the WFS's own flat-wavefront reference intensity, which is subtracted from every subsequent frame passed through `ProcessFrame`.

In [ ]:
# frame processor creation
framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

## Instantiating the reconstructor

We build the network and (if present) resume from a previously trained checkpoint further below, once the `Trainer` exists — see the "The Trainer" section.

In [ ]:
# Phase reconstructor
phaseReconstructor = Papyrus2ndStage(DMParams).to(device = device)

total_params = sum(p.numel() for p in phaseReconstructor.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")


## Optimizer and loss

We optimize only the reconstructor's parameters, with AdamW. `LogResidualVarianceLoss` is a physics-aware loss: it computes `ln(var(residual phase over the pupil))`, i.e. the log-variance of the wavefront error in radians², which is directly tied to the residual RMS/Strehl ratio the AO loop would achieve — not just an abstract regression loss on the mode coefficients.

In [ ]:
# Optimization parameters (learning rate lr and nb of runs)
lrn = TrainParams['lrn']
num_iterations = 1 # Closed loop iterations

# Number of training and testing run 
TrainRunNb = TrainParams['TrainRunNb']
optimizer_n = torch.optim.AdamW(phaseReconstructor.parameters(), lrn, fused = True)

# Setting the loss
loss_variance = LogResidualVarianceLoss(dataset.pupil)

## The Trainer

`AI4AO.Trainer.Trainer` bundles the WFS, DM, frame preprocessor, modal basis (`M2C`), reconstructor, dataset, loss and optimizer, and implements the closed-loop training step (see `AI4AO/Trainer.py`). It also exposes `save_checkpoint`/`load_checkpoint` for persisting reconstructor + optimizer state, and the `evaluate`/`plot_losses` helpers used further down.

We try to resume from a previous checkpoint before training further. Note: an `OziriisCNNOnSky.pth` saved by the older `torch.save(phaseReconstructor.state_dict(), ...)` pattern is a raw state dict, not the wrapped format `save_checkpoint` writes, so loading it here will raise `KeyError` and fall back to training from scratch — re-save once with `trainer.save_checkpoint(...)` to make it loadable by `load_checkpoint` going forward.

In [ ]:
trainer = Trainer(wfs=wfs,
                  framePreprocessor=framePreprocessor,
                  dm=dm,
                  M2C=M2C,
                  phaseReconstructor=phaseReconstructor,
                  dataset=dataset,
                  loss=loss_variance,
                  optimizer=optimizer_n)

try:
    trainer.load_checkpoint(PATH + "OziriisCNNOnSky.pth", load_optimizer = False)
except KeyError:
    # Existing checkpoint predates save_checkpoint's format (a raw state_dict);
    # once re-saved with trainer.save_checkpoint it will load cleanly here.
    print("Starting from scratch")

## Training

`trainer.train(training_steps, closed_loop_iterations)` runs `training_steps` closed-loop optimizer updates. `closed_loop_iterations` sets how many AO-loop steps are simulated — and backpropagated through — per optimizer update; with more than 1, the reconstructor is trained to perform well *given* its own previous corrections, rather than only on independent open-loop frames.

It returns two per-step loss trackers: `loss_tracker`, the network's actual training loss, and `loss_tracker_ideal`, the loss that would result from a perfect projection of the true residual phase onto the modal basis instead of the network's prediction — a lower bound to compare against.

In [ ]:
TrainRunNb = 5000
num_iterations = 1

loss_tracker, loss_tracker_ideal = trainer.train(TrainRunNb, num_iterations)

`trainer.plot_losses` smooths and plots both trackers together. The gap between the training loss and the ideal-loss lower bound indicates how much reconstruction performance is still on the table for the network to gain, versus how much is fundamental to the chosen modal basis and WFS.

In [ ]:
trainer.plot_losses(loss_tracker, loss_tracker_ideal, log_x = True, ylim = (-3, 1))

## Saving

Persist the trained reconstructor, along with the optimizer state (for resuming later), to disk via `trainer.save_checkpoint`.

In [ ]:
trainer.save_checkpoint(PATH + "OziriisCNNOnSky.pth")

## Visualizing a closed loop

`trainer.evaluate()` runs a no-grad closed-loop rollout (reconstructor in `.eval()` mode, no pupil noise injected) and returns an `EvaluationResult` holding the phase, pupil, reconstructed phase, residual phase and WFS frames at every simulated step, ready to animate.

We turn scintillation back off and increase `dataset.Nphases` here only to get a fresh, longer sequence of correlated phase screens (recall `generateClosedLoop = True` from earlier) for the animation — this does not affect the trained weights.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

AtmosParams_eval = AtmosParams.copy()
AtmosParams_eval['Scintillation'] = False
# Dataset creation
dataset_eval = PhaseDataset(WFSParams, AtmosParams_eval, LoopParams, DMParams, device)
dataset_eval.generateClosedLoop = True
dataset_eval.Nphases = 9

print("Starting the optical propagation")
n_frames = 50
result = trainer.evaluate(n_steps = n_frames, dataset = dataset_eval)
print("Finished optical propagation")

print("Starting with display...")
fig, axes = imshow_multiple(
    [
        result.pupil[0],
        result.phase[0],
        result.phase_reconstructed[0],
        result.residual_phase[0],
        result.wfs_frames[0],
    ],
    same_scale=False
)


def update(i):
    imshow_multiple(
        [
            result.pupil[i],
            result.phase[i],
            result.phase_reconstructed[i],
            result.residual_phase[i],
            result.wfs_frames[i],
        ],
        fig=fig,
        axes=axes,
        same_scale=False
    )

    return [
        ax.images[0]
        for tensor_axes in axes
        for ax in tensor_axes
    ]

anim = FuncAnimation(
    fig,
    update,
    frames=n_frames,
    interval=50,
    blit=True
)

# plt.show()
plt.close(fig)

       
# HTML(anim.to_html5_video())
mpl.rcParams["animation.embed_limit"] = 100
HTML(anim.to_jshtml())